In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Comparar configurações do transpilador

<Accordion>
<AccordionItem title="Versões dos pacotes">

O código nesta página foi desenvolvido usando os seguintes requisitos.
Recomendamos usar estas versões ou mais recentes.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Diferentes configurações do transpilador oferecem diferentes tipos de otimização para o circuito, muitas vezes à custa de um tempo de processamento clássico mais longo. Este guia percorre todo o processo de criação, transpilação e envio de circuitos para demonstrar como testar o desempenho de diversas configurações.

Observe que a mesma configuração pode melhorar os resultados de um circuito e prejudicar outro. Certifique-se de inspecionar os circuitos transpilados resultantes antes de executá-los em hardware real.
## Configurar e criar circuito de exemplo

In [1]:
# Create circuit to test transpiler on
from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.circuit.library import grover_operator, DiagonalGate

# Use Statevector object to calculate the ideal output
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram
from qiskit.transpiler import PassManager

from qiskit.circuit.library import XGate
from qiskit.quantum_info import hellinger_fidelity

Crie um pequeno circuito para o transpilador tentar otimizar. Este exemplo cria um circuito que executa o algoritmo de Grover com um oráculo que marca o estado `111`. Em seguida, simule a distribuição ideal (o que você esperaria medir se executasse isso em um computador quântico perfeito um número infinito de vezes) para comparação posterior.

In [2]:
oracle = DiagonalGate([1] * 7 + [-1])
qc = QuantumCircuit(3)
qc.h([0, 1, 2])
qc = qc.compose(grover_operator(oracle))

qc.draw(output="mpl", style="iqp")

<Image src="../docs/images/guides/circuit-transpilation-settings/extracted-outputs/4ac958d4-b9b5-4939-a359-a9edca7ddb6a-0.svg" alt="Output of the previous code cell" />

In [3]:
ideal_distribution = Statevector.from_instruction(qc).probabilities_dict()

plot_histogram(ideal_distribution)

<Image src="../docs/images/guides/circuit-transpilation-settings/extracted-outputs/6313186e-bc40-432e-9ada-8594d6a26d55-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/circuit-transpilation-settings/extracted-outputs/6313186e-bc40-432e-9ada-8594d6a26d55-0.svg)

## Transpilar
Em seguida, transpile os circuitos para a QPU. Você vai comparar o desempenho do transpilador com `optimization_level` definido como `0` (mais baixo) contra `3` (mais alto). O nível de otimização mais baixo faz o mínimo necessário para que o circuito rode no dispositivo: mapeia os qubits do circuito para os qubits do dispositivo e adiciona portas swap para permitir todas as operações de dois qubits. O nível de otimização mais alto é muito mais inteligente e usa várias técnicas para reduzir a contagem total de portas. Como as portas de múltiplos qubits têm altas taxas de erro e os qubits decoerêm com o tempo, os circuitos mais curtos devem produzir melhores resultados.

> **Important:** Este exemplo usa hardware IBM Quantum&reg;, mas você pode experimentá-lo em qualquer QPU compatível com Qiskit. Seus resultados podem ser diferentes.

A célula a seguir transpila `qc` para ambos os valores de `optimization_level`, imprime o número de portas de dois qubits e adiciona os circuitos transpilados a uma lista. Alguns algoritmos do transpilador são aleatorizados, portanto, é definida uma semente para reprodutibilidade.

In [4]:
# Use Qiskit Runtime to run jobs on hardware
from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    SamplerV2 as Sampler,
)

In [5]:
# Select the backend with the fewest number of jobs in the queue
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)
backend.name

'ibm_marrakesh'

In [6]:
# Need to add measurements to the circuit
qc.measure_all()

# Find the correct two-qubit gate
twoQ_gates = set(["ecr", "cz", "cx"])
for gate in backend.basis_gates:
    if gate in twoQ_gates:
        twoQ_gate = gate

circuits = []
for optimization_level in [0, 3]:
    pm = generate_preset_pass_manager(
        optimization_level, backend=backend, seed_transpiler=0
    )
    t_qc = pm.run(qc)
    print(
        f"Two-qubit gates (optimization_level={optimization_level}): ",
        t_qc.count_ops()[twoQ_gate],
    )
    circuits.append(t_qc)

Two-qubit gates (optimization_level=0):  21
Two-qubit gates (optimization_level=3):  12


Since CNOTs usually have a high error rate, the circuit transpiled with `optimization_level=3` should perform much better.

Another way you can improve performance is through [dynamical decoupling](/docs/api/qiskit/qiskit.transpiler.passes.PadDynamicalDecoupling), by applying a sequence of gates to idling qubits. This cancels out some unwanted interactions with the environment. The following cell adds dynamic decoupling to the circuit transpiled with `optimization_level=3` and adds it to the list.

In [7]:
from qiskit_ibm_runtime.transpiler.passes.scheduling import (
    ASAPScheduleAnalysis,
    PadDynamicalDecoupling,
)

# Get gate durations so the transpiler knows how long each operation takes
durations = backend.target.durations()

# This is the sequence we'll apply to idling qubits
dd_sequence = [XGate(), XGate()]

# Run scheduling and dynamic decoupling passes on circuit
pm = PassManager(
    [
        ASAPScheduleAnalysis(durations),
        PadDynamicalDecoupling(durations, dd_sequence),
    ]
)
circ_dd = pm.run(circuits[1])

# Add this new circuit to our list
circuits.append(circ_dd)

In [8]:
circ_dd.draw(output="mpl", style="iqp", idle_wires=False)

<Image src="../docs/images/guides/circuit-transpilation-settings/extracted-outputs/c1c91fbd-acfe-413e-a6c9-ad97f4dd5543-0.svg" alt="Output of the previous code cell" />

Como os CNOTs geralmente têm uma alta taxa de erro, o circuito transpilado com `optimization_level=3` deve ter um desempenho muito melhor.

Outra maneira de melhorar o desempenho é por meio do [desacoplamento dinâmico](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.PadDynamicalDecoupling), aplicando uma sequência de portas a qubits ociosos. Isso cancela algumas interações indesejadas com o ambiente. A célula a seguir adiciona o desacoplamento dinâmico ao circuito transpilado com `optimization_level=3` e o adiciona à lista.

In [9]:
sampler = Sampler(backend)
job = sampler.run(
    [(circuit) for circuit in circuits],  # sample all three circuits
    shots=8000,
)
result = job.result()

## View results

Finally, plot the results from the device runs against the ideal distribution. You can see the results with `optimization_level=3` are closer to the ideal distribution due to the lower gate count, and `optimization_level=3 + dd` is even closer due to the dynamical decoupling.

In [10]:
binary_prob = [
    {
        k: v / res.data.meas.num_shots
        for k, v in res.data.meas.get_counts().items()
    }
    for res in result
]
plot_histogram(
    binary_prob + [ideal_distribution],
    bar_labels=False,
    legend=[
        "optimization_level=0",
        "optimization_level=3",
        "optimization_level=3 + dd",
        "ideal distribution",
    ],
)

<Image src="../docs/images/guides/circuit-transpilation-settings/extracted-outputs/9e86132d-a8b2-40db-af42-53042dfa108b-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/circuit-transpilation-settings/extracted-outputs/c1c91fbd-acfe-413e-a6c9-ad97f4dd5543-0.svg)

## Executar o circuito
Neste ponto, você tem uma lista de circuitos transpilados com diferentes configurações. Em seguida, execute esses circuitos usando o primitivo Sampler e armazene os resultados em `result`.

In [11]:
for prob in binary_prob:
    print(f"{hellinger_fidelity(prob, ideal_distribution):.3f}")

0.985
0.989
0.988


## Ver resultados
Por fim, plote os resultados das execuções no dispositivo em relação à distribuição ideal. Você pode ver que os resultados com `optimization_level=3` estão mais próximos da distribuição ideal devido à menor contagem de portas, e `optimization_level=3 + dd` está ainda mais próximo devido ao desacoplamento dinâmico.